In [43]:
# Headstone_Extraction.py
# 5/5/2026
# Extracts headstone solder card location information

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json
import sys
import os
import random
import requests
import math
import numpy as np
import urllib3
import inspect

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print('\n')
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
        print('\n')
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

def get_tic():
    tic=datetime.datetime.now()
    return tic

def fill_in_missing(jwb_full_df,field_list):
    field_list_set=set(field_list)
    jwb_full_df_set=set(list(jwb_full_df))
    missing_columns=field_list_set-jwb_full_df_set
    jwb_full_df[list(missing_columns)]=np.nan
    return jwb_full_df

# def download_jwb_image(url,image_name):
#     global my_header_image
#     dir_image='C:\\Users\\jrozm\\OneDrive\\Benjamin\\0-Master Worksheets\\Ancestry\\Photos\\'
#     imageOut=dir_image+image_name+'.jpg'
#     with requests.get(url, headers=my_header_image, stream=True, allow_redirects=True) as r:
#         r.raise_for_status()
#         with open(imageOut, "wb") as f:
#             for chunk in r.iter_content(chunk_size=1024 * 256):
#                 if chunk:
#                     f.write(chunk)
#     return imageOut


def download_jwb_image(url, image_name):
    global my_header_image

    dir_image = r"C:\Users\jrozm\OneDrive\Benjamin\0-Master Worksheets\Ancestry\Photos"
    os.makedirs(dir_image, exist_ok=True)

    base_path = os.path.join(dir_image, image_name)
    imageOut = base_path + ".jpg"

    # If file exists, add (1), (2), (3)...
    if os.path.exists(imageOut):
        i = 1
        while True:
            candidate = f"{base_path} ({i}).jpg"
            if not os.path.exists(candidate):
                imageOut = candidate
                break
            i += 1

    with requests.get(url, headers=my_header_image, stream=True, allow_redirects=True) as r:
        r.raise_for_status()
        with open(imageOut, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)

    return os.path.basename(imageOut)

def load_no_match_line(no_match_df,fileOutInt):
    global jwb_full_header,jwb_df,jwb_full_df
    jwb_df = pd.concat([jwb_df,no_match_df])
    jwb_df.reset_index(drop=True, inplace = True)
    jwb_df.to_csv(fileOutInt,mode="w", index=False,header=True)
    return jwb_df

def haystack_contains_needle(needle_df,haystack_df):
    mask = [
    parent in spouse
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask

def all_parts_in(needle, haystack):
    if pd.isna(haystack) or pd.isna(needle):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    match=haystack+'_match' 
    dataframe_df=dataframe_df.copy()
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def find_name_match(dataframe_df,needle,needle_2,haystack):
    out_df,mask=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    if not any(mask):
        names_length=dataframe_df[haystack].str.split(' ').str.len()
        needle_df2=jwb_expand_df[needle_2]
        mask2=haystack_contains_needle(needle_df2,dataframe_df)
        name_match=(mask2 and names_length<3)
        jwb_name_match='Full_2'
    else:
        name_match=mask
        jwb_name_match='Full'
    return jwb_expand_df[name_match],jwb_name_match



my_header_image = {
 'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
'accept-encoding': 'gzip, deflate, br, zstd',
'accept-language': 'en-GB,en-US;q=0.9,en;q=0.8,ja;q=0.7',
'cache-control': 'no-cache',
'pragma': 'no-cache',
'priority': 'u=0, i',
'referer': 'https://www.ancestry.com/search/?searchMode=advanced&searchOrigin=navigation_header',
'sec-ch-ua': '"Google Chrome";v="147", "Not.A/Brand";v="8", "Chromium";v="147"',
'sec-ch-ua-mobile': '?0',
'sec-ch-ua-platform': '"Windows"',
'sec-fetch-dest': 'document',
'sec-fetch-mode': 'navigate',
'sec-fetch-site': 'same-origin',
'sec-fetch-user': '?1',
'upgrade-insecure-requests': '1',
'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'
}

# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++
blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Jewish Welfare Board Card Find Missing Parents'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++

# +++++++++++++++++++ File Input and Output +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\0-Master Worksheets\\Ancestry\\"

fileInput="JWB_Missing_Parents_Input"
fileOutput="Headstone_Index"


fileIn=DirMain+"Inputs\\"+fileInput+".csv"
fileOut=DirMain+"Outputs\\"+fileOutput+"_"+timestamp+".csv"
fileOutInt=DirMain+"Outputs\\"+fileOutput+"_"+timestamp+"_Interim.csv"

tic_start_total=get_tic()
tic_start=get_tic()
tic=get_tic()    

jwb_missing_input_df=pd.read_csv(fileIn, encoding='ISO-8859-1')
total_name_count=len(jwb_missing_input_df)

sleep_range=[8,15] 
sleep_time_pause=1*60
pause_count=100
get_counter=0

url_first_name='https://www.ancestry.com/search/?name='
url_death_year='&death='
url_suffix='&count=50&death_x=0-0-0&name_x=1_1&priority=jewish&searchMode=advanced'

headstone_df=pd.DataFrame([])
name_count=0
for index, row in jwb_missing_input_df.iterrows():
    name_count+=1
    name_index=row['name_index'] 
    last_name=row['last_name'] 
    first_name=row['first_name'] 
    middle_initial=row['middle_initial'] 
    death_year=row['death_year'] 
    
    time_now_obj=datetime.datetime.now()
    time_now=time_now_obj.strftime('%H:%M:%S')  

    # print(f" *** {name_index} • {jwb_name_match} •• {entry_state}  •• Record Id: {record_id} -  Name Count: {name_count}/{total_name_count}  • Time: {time_now}") 
    if middle_initial=='NMN':
        url=url_first_name+first_name+'_'+last_name+url_death_year+str(death_year)+url_suffix
    else:
        url=url_first_name+first_name+'+'+middle_initial+'_'+last_name+url_death_year+str(death_year)+url_suffix
    
    response = requests.request("GET", url, headers=my_header_image)
    status_code=response.status_code
    url_source = response.text
  
    get_counter=random_sleep_new(get_counter) 
    m = re.search(r'"imageViewerUrl":"\/imageviewer\/collections\/9170\/images\/([^\?]+)\?pid=([^\&]+)[^I]+Id\=([^\&]+)',url_source)
    if m is None:
        print('Match Not Found:',name_index)
        headstone_id='None'
        image_id='None'
        query_id='None'
    else:
        image_id=m.group(1)
        headstone_id=m.group(2)
        query_id=m.group(3)
    url_image='https://www.ancestry.com/imageviewer/collections/9170/images/' +image_id+'?queryId='+query_id+'&pId='+str(headstone_id)
    headstone_line_df=pd.DataFrame([name_index,last_name,first_name,middle_initial,death_year,headstone_id,image_id,query_id,url,url_image]).T
    headstone_line_df_header=['name_index','last_name','first_name','middle_initial','death_year','headstone_id','image_id','query_id','url_refer','url_image'] 
    headstone_line_df.columns=headstone_line_df_header
    headstone_df = pd.concat([headstone_df, headstone_line_df])
    headstone_df.to_csv(fileOutInt,mode="w", index=False,header=True)   
    process_time= time_lapse(tic)
    process_time_cum= time_lapse(tic_start)
    tic=get_tic()     
    print(f" *** {name_index} •   Name Count: {name_count}/{total_name_count}  •  Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes")         

    
 

    
    # process_time= time_lapse(tic)
    # process_time_cum= time_lapse(tic_start)
    # tic=get_tic() 
    # # print(f" {blank_line}", Lend='\r') 
    # print(f" *** {name_index:<35} • {jwb_name_match:<5} • {rank:<20} •• {entry_state:<15} • Image Location: {jwb_image_file:<35} • Name Count: {name_count}/{total_name_count}•  Process Time: {process_time/60:4.1f}/{process_time_cum/60:4.1f} Minutes") 


       
headstone_df.to_csv(fileOut,mode="w", index=False,header=True)   
print('****** Processing Complete **********',time_now)

**Starting ***  Jewish Welfare Board Card Find Missing Parents  *** , Time: 2026-05-05 17:11:44.286596  **

 *** Canter-Frank-J-19440922 •   Name Count: 1/454  •  Process Time:  0.2/ 0.2 Minutes
 *** Abraham-Andrew-19450409 •   Name Count: 2/454  •  Process Time:  0.2/ 0.4 Minutes
 *** Abraham-Clayton-S-19441223 •   Name Count: 3/454  •  Process Time:  0.1/ 0.6 Minutes
 *** Ackerman-Alden-L-19450101 •   Name Count: 4/454  •  Process Time:  0.2/ 0.8 Minutes
 *** Albers-John-L-19441007 •   Name Count: 5/454  •  Process Time:  0.2/ 1.0 Minutes
 *** Albert-August-A-19441111 •   Name Count: 6/454  •  Process Time:  0.2/ 1.2 Minutes
 *** Albert-Thomas-A-19431105 •   Name Count: 7/454  •  Process Time:  0.2/ 1.5 Minutes
 *** Aliskowitch-Stanley-J-19460531 •   Name Count: 8/454  •  Process Time:  0.2/ 1.7 Minutes
 *** Alman-Joseph-19440612 •   Name Count: 9/454  •  Process Time:  0.2/ 2.0 Minutes
 *** Andres-Henry-R-19441213 •   Name Count: 10/454  •  Process Time:  0.2/ 2.2 Minutes
 *** Apple

In [40]:
m = re.search(r'"imageViewerUrl":"\/imageviewer\/collections\/9170\/images\/([^\?]+)\?pid=([^\&]+)[^I]+Id\=([^\&]+)',url_source)

m.group(1)
m.group(2)
m.group(3)

'329c555f-79aa-4552-adc6-adcc21706163'